In [ ]:
import os
import re
import numpy as np
import pandas as pd
from scipy.interpolate import interp1d
from astropy.io import fits
from scipy.interpolate import LinearNDInterpolator, NearestNDInterpolator
from joblib import Parallel, delayed
from numpy.polynomial.polynomial import polyval
import matplotlib.pyplot as plt
import astropy.units as u
import phoenix4all
from matplotlib.collections import LineCollection
import matplotlib.ticker as ticker

In [ ]:
class FilterLibrary:
    def __init__(self, FilterPath, VegaPath):
        # initialize paths and zero-point cache
        self.FilterPath = FilterPath
        self.VegaPath = VegaPath
        self.calculated_zps = {}

    def get_filter(self, filterName, system='JWST'):
        # map system names to file prefixes
        system = system.lower()
        system_map = {
            'jwst':      f'JWST_NIRCam.{filterName}.dat',
            'nircam':    f'JWST_NIRCam.{filterName}.dat',
            'jwst_miri': f'JWST_MIRI.{filterName}.dat',
            'miri':      f'JWST_MIRI.{filterName}.dat',
            'spitzer':   f'Spitzer_IRAC.{filterName}.dat',
            '2mass':     f'2MASS_2MASS.{filterName}.dat'
        }
        
        if system not in system_map:
            print(f'Photometric System Not Found: {system}')
            return None
        
        filename = system_map[system]
        full_path = os.path.join(self.FilterPath, filename)
        
        if not os.path.exists(full_path):
            print(f'Filter file missing: {filename}')
            return None
        
        try:
            # load filter data and convert wavelengths to microns
            data = pd.read_csv(full_path, sep=r'\s+', comment='#', header=None, names=['wave', 'trans'], low_memory=False)
            data['wave'] = pd.to_numeric(data['wave'], errors='coerce') / 10000.0
            data['trans'] = pd.to_numeric(data['trans'], errors='coerce')
            
            # return cleaned dataframe
            return data.dropna()
            
        except Exception as e:
            print(f"Error loading Filter Profile: {e}")
            return None

    def get_width_files(self, width, system='JWST'):
        # extract width parameters from filenames
        all_files = os.listdir(self.FilterPath)
        if system.lower() == 'spitzer':
            bandFiles = [f for f in all_files if f.lower().startswith('spitzer')]
            if bandFiles == []:
                print(f'No filters for system: {system}')
                return None
            spitzerFiles = []
            for filename in bandFiles:
                match = re.search(r'(I\d).dat', filename)
                if match:
                    spitzerFiles.append(match.group(1))
            return spitzerFiles

        if system.lower() == '2mass':
            bandFiles = [f for f in all_files if f.lower().startswith('2mass')]
            if bandFiles == []:
                print(f'No filters for system: {system}')
                return None
            TwoMASSFiles = []
            for filename in bandFiles:
                match = re.search(r'.([A-Za-z]*).dat', filename)
                if match:
                    TwoMASSFiles.append(match.group(1))
            return TwoMASSFiles
            
        bandFiles = [f for f in all_files if f.lower().endswith(f'{width.lower()}.dat')]
        if bandFiles == []:
            print(f'No filters with width: {width}')
            return None
        filterNames = []
        for filename in bandFiles:
            match = re.search(r'(F\d+[A-Za-z0-9]*)', filename)
            if match:
                filterNames.append(match.group(1))
        return filterNames

    def get_midpoint(self, filterName, system='JWST'):
        # define midpoints for common filters
        if system.lower() == 'jwst':
            match = re.search(r'F(\d+)', filterName)
            if match:
                return (int(match.group(1)) / 100)
            print(f'No match for: {filterName}')
            return None
        elif system.lower() == 'spitzer':
            if filterName.lower() == 'i1':
                return 3.6
            elif filterName.lower() == 'i2':
                return 4.5
            else:
                print(f'Filter Not Found: {filterName}')
                return None
        elif system.lower() == '2mass':
            if filterName.lower() == 'j':
                midpoint = ((14208.52+10690.53) / 2) * 10**(-4)
                return midpoint
            elif filterName.lower() == 'h':
                midpoint = ((18277.23+14465.17) / 2) * 10**(-4)
                return midpoint
            elif filterName.lower() == 'ks':
                midpoint = ((23810.79+19402.19) / 2) * 10**(-4)
                return midpoint
        else:
            print(f'System Not Found {system}')
            return None

    def get_filters_covering_range(self, wave_min, wave_max):
        # scan directory for matching filters
        all_files = os.listdir(self.FilterPath)
        matches = []
    
        for filename in all_files:
            if not filename.endswith(".dat"):
                continue
    
            # identify system from filename prefix
            if filename.startswith("JWST_NIRCam."):
                system = "JWST"
                filter_name = filename.replace("JWST_NIRCam.", "").replace(".dat", "")
            elif filename.startswith("Spitzer_IRAC."):
                system = "Spitzer"
                filter_name = filename.replace("Spitzer_IRAC.", "").replace(".dat", "")
            elif filename.startswith("2MASS_2MASS."):
                system = "2MASS"
                filter_name = filename.replace("2MASS_2MASS.", "").replace(".dat", "")
            elif filename.startswith("JWST_MIRI."):
                system = "MIRI"
                filter_name = filename.replace("JWST_MIRI.", "").replace(".dat", "")
            else:
                continue
    
            band = self.get_filter(filter_name, system)
            if band is None:
                continue
            
            f_min, f_max = band['wave'].min(), band['wave'].max()
    
            # check if filter overlaps with spectrum
            if f_min >= wave_min and f_max <= wave_max:
                matches.append((system, filter_name))
    
        return matches

    def integrate_filter(self, spectrum_wave, spectrum_flux, filter_name, system='JWST', spectrum_err=None):
        band = self.get_filter(filter_name, system)
        if band is None: 
            print(f'error Band doesnt exist: {filter_name,system}')
            return None, None

        f_min, f_max = band['wave'].min(), band['wave'].max()

        if spectrum_wave.min() > f_min or spectrum_wave.max() < f_max:
            print(f"Warning: Spectrum does not fully cover filter {filter_name}!")
            return None, None

        # interpolate spectrum to match filter wavelengths
        spec_interpolator = interp1d(spectrum_wave, spectrum_flux, kind='linear', fill_value=0.0, bounds_error=False)
        resampled_flux = spec_interpolator(band['wave'])
        
        # perform trapezoidal integration
        numerator = np.trapezoid(resampled_flux * band['trans'] * band['wave'], band['wave'])
        denominator = np.trapezoid(band['trans'] * band['wave'], band['wave'])

        # prevent division by zero
        if denominator == 0: 
            print(f'error denominator: {denominator}, numerator: {numerator}')
            return None, None

        photometry = numerator / denominator

        # interpolate errors if provided
        phot_err = None
        if spectrum_err is not None:
            err_interpolator = interp1d(spectrum_wave, spectrum_err, kind='linear', fill_value=0.0, bounds_error=False)
            resampled_err = err_interpolator(band['wave'])

            err_num = np.sqrt(np.trapezoid((resampled_err * band['trans'] * band['wave'])**2, band['wave']))
            phot_err = err_num / denominator
        
        return photometry, phot_err

    def get_vega_zp(self, fName, system='JWST'):
        # cache vega zero points to save time
        if fName in self.calculated_zps:
            return self.calculated_zps[fName]
    
        vega_path = self.VegaPath
        try:  
            with fits.open(vega_path) as hdul:
                data = hdul[1].data
            
                vega_wave_ang = data['WAVELENGTH']
                vega_flux_cgs = data['FLUX']
                vega_wave_microns = vega_wave_ang / 10000.0

                zp_val, _ = self.integrate_filter(vega_wave_microns, vega_flux_cgs, fName, system)

                if zp_val is not None:
                    self.calculated_zps[fName] = zp_val
                    return zp_val
                else:
                    print(f"Failed to calculate ZP for {fName}")
                    return None
                
        except Exception as e:
            print(f"Error loading Vega file: {e}")
            return None

    def get_magnitude(self, fName, flux, radius, system='JWST'):
        # calculate synthetic magnitude for theoretical models
        R = 695700 * radius 
        d = 3.086e14 
        zeroPoint = self.get_vega_zp(fName, system)
        
        if zeroPoint is None:
            return None
            
        insideLog = (flux*(R/d)**2) / zeroPoint
        magnitude = - 2.5 * np.log10(insideLog)
        return magnitude

    def get_observed_magnitude(self, fName, observed_flux, distance, system='JWST', observed_err=None):
        # calculate magnitude for data already observed at earth
        zeroPoint = self.get_vega_zp(fName, system)
        
        if zeroPoint is None or observed_flux <= 0:
            return {'M': np.nan, 'upperr': np.nan, 'lowerr': np.nan, 'phot_err': np.nan}
            
        magnitude = -2.5 * np.log10(observed_flux / zeroPoint)
        
        # expects distance in kpc for the errors below
        M = magnitude - 5 * np.log10(distance['dist'] / 0.01)
        M_plus = magnitude - 5 * np.log10((distance['dist'] - distance['lowerr']) / 0.01) 
        M_minus = magnitude - 5 * np.log10((distance['dist'] + distance['upperr']) / 0.01)
        
        phot_mag_err = np.nan
        if observed_err is not None and observed_err > 0:
            phot_mag_err = 1.0857 * (observed_err / observed_flux)
        
        Mag = {'M': M, 'upperr': M_plus, 'lowerr': M_minus, 'phot_err': phot_mag_err}
        return Mag

# set relative paths for github structure
filters_path = "./data/filters"
vega_path = "./data/empirical/alpha_lyr_stis_011.fits"

Filter_lib = FilterLibrary(FilterPath=filters_path, VegaPath=vega_path)

In [ ]:
class SpectralLibrary:
    def __init__(self, sonora_path, photometry_path):
        # initialize paths and load the pre-calculated photometry
        self.sonora_path = sonora_path
        self.photometry_path = photometry_path
        
        # load grids for all three metallicities
        self.photometry_pos = self._load_photometry(m='+0.5')
        self.photometry_neg = self._load_photometry(m='-0.5')
        self.photometry = self._load_photometry(m='0.0')

    def get_spectrum(self, teff, logg=5.0, m='0.0'):
        # load sonora bobcat atmospheric model
        return self._load_sonora(teff, logg, m)
            
    def _load_sonora(self, teff, logg, m):
        # format the required file string for sonora bobcat models
        g_val = int(10**(logg - 2))
        if m == '0.0':
            filename = f"sp_t{int(teff)}g{g_val}nc_m0.0.gz"
        else: 
            filename = f"sp_t{int(teff)}g{g_val}nc_m{m}"
            
        folder = f"spectra_m{m}"
        
        if self.sonora_path is None:
            print("Error: Sonora path not set.")
            return None

        partial_path = os.path.join(folder, filename)
        full_path = os.path.join(self.sonora_path, partial_path)

        if not os.path.exists(full_path):
            print(f"Sonora file missing: {partial_path}")
            return None
            
        try:
            # load spectrum and convert surface flux density to cgs units
            data = pd.read_csv(full_path, sep=r'\s+', comment='#', names=['wave', 'flux'], usecols=[0, 1], compression='infer', low_memory=False, skiprows=1)
            data['wave'] = pd.to_numeric(data['wave'], errors='coerce')
            data['flux'] = pd.to_numeric(data['flux'], errors='coerce')
            
            c_angstrom = 2.99792e18 
            wave_angstrom = data['wave'] * 10000 
            data['flux'] = data['flux'] * (c_angstrom / wave_angstrom**2)
            
            return data.dropna()
            
        except Exception as e:
            print(f"Error loading Sonora: {e}")
            return None

    def _load_mass_age(self, m='0.0'):
        # match input metallicity to file path naming convention
        met_id = {'0.0': '+0.0',
                 '+0.0': '+0.0',
                 '0.5': '+0.5',
                 '+0.5': '+0.5',
                 '-0.5': '-0.5'}

        if m not in met_id:
            print(f'Metalicity Not Recognised: {m}')
            return None
    
        met = met_id[m]
        
        # safely build cross-platform file path
        evo_path = os.path.join("evolution_tables", f"nc{met}_co1.0_mass_age")
        file_path = os.path.join(self.sonora_path, evo_path)
        
        # load evolutionary tracks
        df_evo = pd.read_csv(file_path, sep=r"\s+", comment="#", header=None, skiprows=2)
        df_evo.columns = ["Teff", "logg", "mass", "radius", "logL", "age"]

        return df_evo

    def get_radius(self, Teff, logg, m='0.0'):
        # pull data from mass-age tables
        df = self._load_mass_age(m)

        points = df[['Teff', 'logg']].values
        values = df['radius'].values

        # perform 2d interpolation to find exact radius
        linear_interpolator = LinearNDInterpolator(points, values)
        nearest_interp = NearestNDInterpolator(points, values)

        r = linear_interpolator(Teff, logg)

        # fallback to nearest neighbour if out of bounds
        if np.isnan(r):
            r = nearest_interp(Teff, logg)
        
        return float(r)

    def _load_photometry(self, m='0.0'):
        # ensure sonora path is valid before loading
        if self.sonora_path is None:
            print("Error: Sonora path not set.")
            return None
            
        met_id = {'0.0': '0.0',
                 '+0.0': '0.0',
                 '0.5': '+0.5',
                 '+0.5': '+0.5',
                 '-0.5': '-0.5'}

        if m not in met_id:
            print(f'Metalicity Not Recognised: {m}')
            return None
    
        met = met_id[m]
        file_name = f'Sonora Photometry Grid m{m}.csv'
        full_path = os.path.join(self.photometry_path, file_name)
        
        if not os.path.exists(full_path):
            print(f"Sonora Photometry file missing: {file_name}")
            return None

        # read and return the pre-calculated grid
        df = pd.read_csv(full_path)
        return df

    def get_magnitude(self, band, m='0.0'):
        # extract specific filter column from the photometry grid
        df = self._load_photometry(m=m)
        if band not in df.columns:
            print(f'Error: No band named {band} with m = {m}')
            return None
            
        df1 = df.iloc[:, :4].copy()
        df1[band] = df[band]
        return df1

    def get_colour(self, b1, b2, m='0.0'):
        # calculate synthetic colour index
        df1 = self.get_magnitude(b1, m)
        df2 = self.get_magnitude(b2, m)

        colour = df1[b1] - df2[b2]

        df1[b2] = df2[b2]
        df1[f'{b1}-{b2}'] = colour
        return df1

    def make_CMD(self, b1, b2, b3, m='0.0'):
        # generate dataframe ready for plotting on a colour-magnitude diagram
        df_col = self.get_colour(b1, b2, m)
        df_mag = self.get_magnitude(b3, m)

        return df_col, df_mag

# set relative paths for github structure
path_sonora = "./data/models"
photometry_path = "./data/models"

spec_lib = SpectralLibrary(sonora_path=path_sonora, photometry_path=photometry_path)

In [ ]:
class Polynomial_Fits:
    def __init__(self, filePath):
        # initialize path and load the polynomial table automatically
        self.FilePath = filePath
        self.Table = self._load_table()

    def get_magnitudes(self, band):
        # extract coefficients for the specified band and calculate the fit
        df = self.Table
        row = df[df['Band'] == band].squeeze()
        coeffs = [row[f'a{i}'] for i in range(7)]
        s_val = row['s']
        
        # generate a smooth array of spectral types to plot against
        x_fit = np.linspace(row['SpT Min'], row['SpT Max'], 300)
        y_best_fit = polyval(x_fit, coeffs)
        
        return {"mag": y_best_fit,
                "sp_t": x_fit,
                "s": s_val}

    def _load_table(self):
        # load the empirical sequence table safely
        if self.FilePath is None:
            print("Error: Polynomial Fit path not set.")
            return None
        
        if not os.path.exists(self.FilePath):
            print(f"Polynomial Table file missing: {self.FilePath}")
            return None
            
        df = pd.read_csv(self.FilePath)
        return df

    def get_colour(self, band1, band2):
        # subtract two polynomial tracks to get the colour index
        m1 = self.get_magnitudes(band1)
        m2 = self.get_magnitudes(band2)
        
        y1 = np.asarray(m1["mag"]).ravel()
        x1 = np.asarray(m1["sp_t"]).ravel()
        y2 = np.asarray(m2["mag"]).ravel()
        x2 = np.asarray(m2["sp_t"]).ravel()
        
        # find the overlapping valid spectral type range
        x_min = max(x1.min(), x2.min())
        x_max = min(x1.max(), x2.max())
    
        x_common = np.linspace(x_min, x_max, len(x1))
    
        # interpolate both bands onto the common spectral type grid
        y1_common = np.interp(x_common, x1, y1)
        y2_common = np.interp(x_common, x2, y2)
    
        colour = y1_common - y2_common

        s1 = m1["s"]
        s2 = m2["s"]
        # col_upper, col_lower = get_error_lines(s1, s2, colour)
    
        return {"col": colour, "x": x_common}

    def for_CMD(self, band1, band2, mag_band):
        # extract final arrays ready for colour-magnitude diagram plotting
        col = self.get_colour(band1, band2)
        mag = self.get_magnitudes(mag_band)

        x_col = np.asarray(col['x']).ravel()
        y_col = np.asarray(col['col']).ravel()
        mag_x = np.asarray(mag['sp_t']).ravel()
        mag_y = np.asarray(mag['mag']).ravel()

        # ensure colour and magnitude arrays perfectly align
        x_min = max(x_col.min(), mag_x.min())
        x_max = min(x_col.max(), mag_x.max())

        x_common = np.linspace(x_min, x_max, len(mag_x))
        x = np.interp(x_common, x_col, y_col)
        y = np.interp(x_common, mag_x, mag_y)
        
        return x, y, x_common

# load the polynomial equations from the repository data folder
polynomial_path = "./data/empirical/Polynomial_Table.csv"
BD_Poly = Polynomial_Fits(filePath=polynomial_path)

In [ ]:
class PSR_Spectra:
    def __init__(self, FolderPath):
        # initialize variables
        self.FolderPath = FolderPath
        self.fits_file = None
        self.time = None
        self.phase = None
        self.wavelengths = None
        self.flux_ujy = None
        self.flux_err_ujy = None
        self.flux_cgs = None
        self.flux_err_cgs = None
        self.Photometry = None
        self.P = None
        self.T0 = None

    def load_fits(self, fits_filename):
        if self.FolderPath is None:
            print('Error: Folder path not set.')
            return False

        full_path = os.path.join(self.FolderPath, fits_filename)
        if not os.path.exists(full_path):
            print(f"FITS file missing: {fits_filename}")
            return False
        
        try:
            with fits.open(full_path) as hdul:
                self.time = hdul[0].data.astype(float) # in mjd
                self.wavelengths = hdul[1].data.astype(float)  # wavelengths in um
                flux_raw = hdul[2].data.astype(float)  # flux in ujy
                errors_raw = hdul[3].data.astype(float)  # errors in ujy
                
                # quality flags
                q1 = hdul[4].data if len(hdul) > 4 else None
                q2 = hdul[5].data if len(hdul) > 5 else None
                q3 = hdul[6].data if len(hdul) > 6 else None
                
                # apply quality filtering
                mask = np.isnan(flux_raw)
                if q1 is not None: mask = mask | (q1 > 0.2)
                if q2 is not None: mask = mask | (q2 > 0.3)
                if q3 is not None: mask = mask | (q3 > 0.8)
                
                bad_wavelengths = np.all(mask, axis=0)
                self.wavelengths = self.wavelengths[~bad_wavelengths]
                
                flux_clean = flux_raw[:, ~bad_wavelengths]
                errors_clean = errors_raw[:, ~bad_wavelengths]
                
                # transpose to (wavelengths, time) to match wasp-121b architecture exactly
                self.flux_ujy = flux_clean.T
                self.flux_err_ujy = errors_clean.T
                
                self.fits_file = fits_filename
                print(f"Loaded FITS: {fits_filename} | Shape: {self.flux_ujy.shape}")
                
                return True
        except Exception as e:
            print(f"Error loading FITS file: {e}")
            return False

    def set_phase(self, P, T0):
        # save period and epoch so binning function can use them later
        self.P = P
        self.T0 = T0

        phases = (self.time - T0) / P
        phases = phases - 0.25
        phases = phases % 1.0
        phases = np.where(phases > 0.5, phases - 1.0, phases) # wraps around 0.0

        # saved as a dictionary to perfectly match the wasp-121b plotting structure
        self.phase = {'p': phases}

    def set_distance(self, dist_kpc, err_upper, err_lower):
        # calculate parallax in mas based on input distance
        plx_mas = 1000 / (dist_kpc * 1000)
        plx_mas_err = (1000 / (dist_kpc**2 * 1000)) * err_upper 
        
        self.distance = {
            'dist': dist_kpc, 
            'upperr': err_upper, 
            'lowerr': err_lower, 
            'plx': plx_mas, 
            'plx_err': plx_mas_err
        }

    def set_radius(self, r, err):
        self.planet_radius = r
        self.planet_radius_err = err

    def bin_phase_curve(self, nbins=50):
        print(f"Binning data into {nbins} phase bins...")
        time_flat = self.time.flatten()
        bins = np.linspace(time_flat.min(), time_flat.max(), nbins + 1)
        indices = np.digitize(time_flat, bins)

        new_time = []
        new_flux = []
        new_err = []

        for i in range(1, len(bins)):
            mask = (indices == i)
            if np.sum(mask) > 0:
                new_time.append(np.mean(time_flat[mask]))

                f_slice = self.flux_ujy[:, mask]
                e_slice = self.flux_err_ujy[:, mask]

                # weighted mean for accurate noise propagation
                weights = 1.0 / (e_slice ** 2)
                weighted_flux = np.sum(f_slice * weights, axis=1) / np.sum(weights, axis=1)
                weighted_err = np.sqrt(1.0 / np.sum(weights, axis=1))

                new_flux.append(weighted_flux)
                new_err.append(weighted_err)

        self.time = np.array(new_time)
        self.flux_ujy = np.column_stack(new_flux)
        self.flux_err_ujy = np.column_stack(new_err)

        # recalculate phases for the new binned time array using saved variables
        self.set_phase(self.P, self.T0)
        print(f'Completed Binning. New phase curve length: {len(self.time)}')

    def convert_to_cgs(self):
        # vectorized conversion of the entire 2d flux array
        wave_ang = self.wavelengths * 1e4
        c_ang = 2.99792e18  # angstrom s^-1
        
        # calculate the conversion factor for each wavelength
        factor = c_ang / (wave_ang**2) 
        
        # broadcast the factor across the 2d array: (wavelengths, phases)
        f_nu = self.flux_ujy * 1e-29
        e_nu = self.flux_err_ujy * 1e-29
        
        self.flux_cgs = f_nu * factor[:, np.newaxis]
        self.flux_err_cgs = e_nu * factor[:, np.newaxis]
        print("Converted phase curve array to cgs units (erg s-1 cm-2 A-1).")

    def _process_single_filter(self, fName, system):
        mags = []
        mag_errs = []
        
        for i in range(len(self.time)):
            current_flux = self.flux_cgs[:, i]
            current_err = self.flux_err_cgs[:, i]

            # integrate through filter_lib
            flux, flux_err = Filter_lib.integrate_filter(self.wavelengths, current_flux, fName, system, spectrum_err=current_err)
            
            # 2-sigma snr cutoff
            if flux is not None and flux > 0:
                snr = flux / flux_err
                if snr > 2.0:
                    # calculates absolute magnitude natively
                    mag = Filter_lib.get_observed_magnitude(fName, flux, self.distance, system, observed_err=flux_err)
                    mags.append(mag['M'])
                    mag_errs.append(mag['phot_err'])
                else:
                    mags.append(np.nan)
                    mag_errs.append(np.nan)
            else:
                mags.append(np.nan)
                mag_errs.append(np.nan)
                
        return fName, mags, mag_errs

    def run_synthetic_photometry(self, filter_tuples):
        if self.flux_cgs is None:
            print("Error: Run convert_to_cgs() before photometry.")
            return None
            
        print(f"Parallel processing {len(self.time)} PSR spectra through {len(filter_tuples)} filters...")

        results = Parallel(n_jobs=-1)(
            delayed(self._process_single_filter)(fName, system) 
            for system, fName in filter_tuples
        )
        
        df = pd.DataFrame({'Time': self.phase['p']})
        
        for fName, mags, mag_errs in results:
            df[f'Mag_{fName}'] = mags
            df[f'Mag_{fName}_err'] = mag_errs
            
        self.Photometry = df
        return df

    def get_filter_names(self):
        if self.Photometry is None:
            return None
        filter_list = [col.replace('Mag_', '') for col in self.Photometry.columns if col.startswith('Mag_') and not col.endswith('_err')]
        return filter_list

    def load_photometry(self, filepath):
        if filepath is None:
            print('Error: No file path found')
            return None

        if not os.path.exists(filepath):
            print(f"Photometry path doesnt work: {filepath}")
            return None
        df = pd.read_csv(filepath)
        self.Photometry = df
        print('Photometry Loaded!!')
        return True

    def get_magnitudes(self, band, scaled=True):
        # identical to wasp-121b structure
        if self.Photometry is None: return None
        try:
            mag = self.Photometry[f'Mag_{band}']
            err = self.Photometry[f'Mag_{band}_err']
            phase = self.Photometry['Time']
        except:
            print(f'ERROR: No filter named {band}')
            return None

        if scaled == True:
            scaled_mag, scaled_err = self.scale_magnitudes_to_BD(mag, err)
            return {'mag': scaled_mag, 'err': scaled_err, 'phase': phase}

        return {'mag': mag, 'err': err, 'phase': phase}

    def get_colour(self, b1, b2, scaled=True):
        m1 = self.get_magnitudes(b1, scaled)
        m2 = self.get_magnitudes(b2, scaled)
        if m1 is None or m2 is None: return None

        col = m1['mag'] - m2['mag']
        col_err = np.sqrt(m1['err']**2 + m2['err']**2)
        return {'col': col, 'err': col_err, 'phase': m1['phase']}

    def get_CMD(self, b1, b2, b3, scaled=True):
        col = self.get_colour(b1, b2, scaled)
        mag = self.get_magnitudes(b3, scaled)
        if col is None or mag is None: return None

        return {'phase': col['phase'], 'col': col['col'], 'mag': mag['mag'], 'col_err': col['err'], 'mag_err': mag['err']}

    def scale_magnitudes_to_BD(self, mag, err):
        R_p = self.planet_radius
        R_p_err = self.planet_radius_err
        R_BD = 0.9

        mag_offset = 5 * np.log10(R_p/R_BD)

        scaled_mag = mag + mag_offset
        
        radius_err = (5 / np.log(10)) * (R_p_err / R_p)
        scaled_mag_err = np.sqrt(err**2 + radius_err**2)
        
        return scaled_mag, scaled_mag_err

In [ ]:
# 1. load and prepare
folder_path = "./data/phase_curves"
fits_file ="jw05263001001_nrs1_allspec1d.fits"

psr = PSR_Spectra(Folder_Path=folder_path)
psr.load_fits(fits_filename=fits_file)

# load the pre-calculated dataset
photometry_path = './data/phase_curves/PSR_J2322_Photometry_30_bins.csv'
psr.load_photometry(filepath=photometry_path)

# pass the psr-specific variables directly into the new functions
psr.set_phase(P=0.322963997, T0=59692.9699957)
psr.set_distance(dist_kpc=0.630, err_upper=0.075, err_lower=0.060)
psr.set_radius(r=1.1, err=0.1)

# 2. bin down to 30 phases
psr.bin_phase_curve(nbins=30)

# 3. convert units
psr.convert_to_cgs()

In [ ]:
# run synthetic photometry for PSR J2322-2650b

min_lambda = psr.wavelengths.min()
max_lambda = psr.wavelengths.max()

filter_tuples = Filter_lib.get_filters_covering_range(min_lambda, max_lambda)

df_photometry = psr.run_synthetic_photometry(filter_tuples)
df_photometry = df_photometry.sort_values(by='Time').reset_index(drop=True)
#df_photometry.to_csv("./data/phase_curves/PSR_J2322_Photometry_30_bins.csv", index=False)

In [ ]:
class Process_Phase_Curves:
    def __init__(self, filePath):
        # initialize variables
        self.FilePath = filePath
        self.Photometry = None
        self.SecEclipse = None
        self.period = None
        self.transit = None
        
    def _load_data(self, fits=False):
        if self.FilePath is None:
            print('Error: No file path found')
            return None

        if not os.path.exists(self.FilePath):
            print(f"Phase Curve file missing: {self.FilePath}")
            return None

        data = np.load(self.FilePath)
        return data

    def get_data(self):
        # extract raw arrays from the .npz file
        data = self._load_data()
        self.time = data['time']
        self.wavelengths = data['wavelengths']
        self.flux_ratios = data['flux_ratios']
        self.flux_ratios_err = data['flux_ratios_err']

    def set_distance(self, plx_mas, plx_mas_err):
        # convert parallax to distance in kpc
        distance_pc = 1000 / plx_mas
        distance_pc_err = abs((1000 / plx_mas**2) * plx_mas_err)
        dist_kpc = distance_pc * 0.001
        dist_kpc_err = distance_pc_err * 0.001
        self.distance = {'dist': dist_kpc, 'upperr': dist_kpc_err, 'lowerr': dist_kpc_err, 'plx': plx_mas, 'plx_err': plx_mas_err}

    def set_phase(self, P, P_err, t0, t0_err):
        # store period and transit epoch so binning function can reuse them
        self.period = {'P': P, 'P_err': P_err}
        self.transit = {'t0': t0, 't0_err': t0_err}
        
        # convert time to orbital phase
        phase = (self.time - self.transit['t0']) / self.period['P']
        phase_err = self.period['P']**(-1) * np.sqrt(self.transit['t0_err']**2 + (phase**2 * self.period['P_err']))
        self.phase = {'p': phase, 'p_err': phase_err}

    def set_radius(self, radius, radius_err):
        self.planet_radius = radius
        self.planet_radius_err = radius_err

    def get_stellar_flux(self, wave_microns, Teff, Teff_err, R, R_err):
        # physical constants (cgs)
        h = 6.626e-27  # erg s
        c = 2.9979e10  # cm/s
        k = 1.38e-16   # erg/K

        # wavelength to cm
        lam_cm = wave_microns * 1e-4

        # planck function for surface intensity (erg/s/cm2/sr/cm)
        exp = np.exp(h * c / (lam_cm * k * Teff))
        B_lam = (2 * h * c**2 / lam_cm**5) / (exp - 1)

        dB_T = B_lam * (h * c / (lam_cm * k * Teff**2)) * (exp / (exp - 1))
        B_lam_err = abs(dB_T * Teff_err)

        # convert to flux at surface
        F_surface = np.pi * B_lam
        F_surface_err = np.pi * B_lam_err

        # convert wavelength from cm to per angstrom
        F_surface_ang = F_surface / 1e8
        F_surface_ang_err = F_surface_err / 1e8

        # dilute the flux over the distance to earth
        R_star_cm = R * 6.957e10
        R_star_cm_err = R_err * 6.957e10

        plx_mas = self.distance['plx']
        plx_mas_err = self.distance['plx_err']
        distance_pc = 1000 / plx_mas
        distance_pc_err = abs((1000 / plx_mas**2) * plx_mas_err)
        dist_cm = distance_pc * 3.086e18
        dist_cm_err = distance_pc_err * 3.086e18

        dilution = (R_star_cm / dist_cm)**2
        dilution_err = np.sqrt((2 * R_star_cm * R_star_cm_err / dist_cm**2)**2 + (2 * R_star_cm**2 * dist_cm_err / dist_cm**3)**2)

        F_obs = F_surface_ang * dilution
        F_obs_err = np.sqrt((dilution * F_surface_ang_err)**2 + (F_surface_ang * dilution_err)**2)

        return F_obs, F_obs_err

    def phoenix_models(self, wave_microns, logg, feh, alpha, Teff, R):
        fp_fs = self.flux_ratios
        
        # get the model
        lam_model, flux_model = phoenix4all.get_spectrum(source="synphot", teff=Teff, logg=logg, feh=feh, alpha=alpha)
        lam_micron = lam_model.to(u.um)
        
        # distance and radius conversions
        dist = self.distance['dist'] * 1000 # distance in parsecs
        Rstar = R * 6.957e10 # converting to cm from solar radius
        dist_star = dist * 3.086e18 # parsecs to cm 
        
        # interpolate the model onto your specific wavelength grid
        flux_model_ang = flux_model.value
        stellar_flux = np.interp(wave_microns, lam_micron.value, flux_model_ang)
        
        # dilute to earth
        flux_at_earth = stellar_flux * (Rstar / dist_star)**2
        
        # fix the shape mismatch so the 1d flux multiplies across the 2d time bins correctly
        planet_flux = fp_fs * flux_at_earth[:, np.newaxis]
        
        return flux_at_earth, planet_flux

    def get_spectroscopy(self, b1, b2, Teff, Teff_err, R, R_err, system='JWST'):
        mags_b1 = []
        mags_b2 = []
        valid_phases = []
        
        star_flux_obs, star_flux_obs_err = self.get_stellar_flux(self.wavelengths, Teff, Teff_err, R, R_err)
        
        print(f"Integrating {len(self.time)} spectra through {b1} and {b2}")

        for i, phase in enumerate(self.time):
            fp_fs_ratio = self.flux_ratios[:, i]
            fp_fs_ratio_err = self.flux_ratios_err[:, i]
            
            planet_flux_obs = fp_fs_ratio * star_flux_obs
            planet_flux_obs_err = planet_flux_obs * np.sqrt((fp_fs_ratio_err / fp_fs_ratio)**2 + (star_flux_obs_err / star_flux_obs)**2)

            flux1, _ = Filter_lib.integrate_filter(self.wavelengths, planet_flux_obs, b1, system)
            flux2, _ = Filter_lib.integrate_filter(self.wavelengths, planet_flux_obs, b2, system)

            if (flux1 is not None and flux1 > 0) and (flux2 is not None and flux2 > 0):
                mag1 = Filter_lib.get_observed_magnitude(b1, flux1, self.distance, system)
                mag2 = Filter_lib.get_observed_magnitude(b2, flux2, self.distance, system)
                
                mags_b1.append(mag1['M'])
                mags_b2.append(mag2['M'])
            else:
                mags_b1.append(np.nan)
                mags_b2.append(np.nan)
                
        print(f"Lengths -> Time: {len(self.time)} | Mag1: {len(mags_b1)} | Mag2: {len(mags_b2)}")
        df = pd.DataFrame({'Time': self.phase['p'], f'Mag_{b1}': mags_b1, f'Mag_{b2}': mags_b2, 'Colour': np.array(mags_b1) - np.array(mags_b2)})
        
        return df

    def _process_single_filter(self, fName, system, planet_flux, planet_flux_err):
        # this function processes all phase magnitudes for 1 filter
        mags = []
        mag_errs = []
        for i in range(len(self.time)):
            current_planet_flux = planet_flux[:, i]
            current_planet_err = planet_flux_err[:, i]

            # integrate
            flux, flux_err = Filter_lib.integrate_filter(self.wavelengths, current_planet_flux, fName, system, spectrum_err=current_planet_err)
            
            # convert to magnitude
            if flux is not None and flux > 0:
                snr = flux / flux_err
                if snr > 2.0:
                    mag = Filter_lib.get_observed_magnitude(fName, flux, self.distance, system, observed_err=flux_err)
                    
                    mags.append(mag['M'])
                    mag_errs.append(mag['phot_err'])
                else:
                    mags.append(np.nan)
                    mag_errs.append(np.nan)
            else:
                mags.append(np.nan)
                mag_errs.append(np.nan)
                
        return fName, mags, mag_errs

    def get_phoenix_spectroscopy(self, filter_tuples, logg, feh, alpha, Teff, R):
        # 1. call your simple phoenix model 
        flux_at_earth, planet_flux = self.phoenix_models(self.wavelengths, logg, feh, alpha, Teff, R)
        
        # 2. calculate the planet flux error using only the telescope noise
        planet_flux_err = self.flux_ratios_err * flux_at_earth[:, np.newaxis]
        
        print(f"Parallel processing {len(self.time)} PHOENIX spectra through {len(filter_tuples)} filters...")

        # 3. run the parallel processing across available cpu cores
        results = Parallel(n_jobs=-1)(
            delayed(self._process_single_filter)(fName, system, planet_flux, planet_flux_err) 
            for system, fName in filter_tuples
        )
        
        # 4. build the final dataframe
        df = pd.DataFrame({'Time': self.phase['p']})
        
        for fName, mags, mag_errs in results:
            df[f'Mag_{fName}'] = mags
            df[f'Mag_{fName}_err'] = mag_errs
            
        return df

    def load_photometry(self, filepath):
        if filepath is None:
            print('Error: No file path found')
            return None

        if not os.path.exists(filepath):
            print(f"Photometry path doesnt work: {filepath}")
            return None
            
        df = pd.read_csv(filepath)
        self.Photometry = df
        print('Photometry Loaded!!')
        return True

    def get_filter_names(self):
        if self.Photometry is None:
            print('Photometry not loaded! RUN "load_photometry"')
            return None
            
        df = self.Photometry
        filter_list = [
            col.replace('Mag_', '')          
            for col in df.columns            
            if col.startswith('Mag_')        
            and not col.endswith('_err')     
        ]
    
        return filter_list

    def get_magnitudes(self, band, scaled=True):
        if self.Photometry is None:
            print('Photometry not loaded! RUN "load_photometry"')
            return None
            
        df = self.Photometry
        try:
            mag = df[f'Mag_{band}']
            err = df[f'Mag_{band}_err']
            phase = df['Time']
        except:
            print(f'ERROR: No filter named {band}')
            return None

        if scaled == True:
            scaled_mag, scaled_err = self.scale_magnitudes_to_BD(mag, err)
            return {'mag': scaled_mag, 'err': scaled_err, 'phase': phase}
            
        return {'mag': mag, 'err': err, 'phase': phase}

    def get_colour(self, b1, b2, scaled=True):
        m1 = self.get_magnitudes(b1, scaled)
        m2 = self.get_magnitudes(b2, scaled)

        if m1 is None or m2 is None:
            return None

        col = m1['mag'] - m2['mag']
        col_err = np.sqrt(m1['err']**2 + m2['err']**2)

        return {'col': col, 'err': col_err, 'phase': m1['phase']}

    def get_CMD(self, b1, b2, b3, scaled=True):
        col = self.get_colour(b1, b2, scaled)
        mag = self.get_magnitudes(b3, scaled)

        if col is None or mag is None:
            return None

        return {'phase': col['phase'], 'col': col['col'], 'mag': mag['mag'], 'col_err': col['err'], 'mag_err': mag['err']}
        
    def bin_phase_curve(self, nbins=50):
        time_flat = self.time.flatten()
        bins = np.linspace(time_flat.min(), time_flat.max(), nbins + 1)
        indices = np.digitize(time_flat, bins)

        new_time = []
        new_flux = []
        new_err = []

        for i in range(1, len(bins)):
            mask = (indices == i)
            if np.sum(mask) > 0:
                # average the time
                new_time.append(np.mean(time_flat[mask]))

                # isolate flux and err for this bin
                f_slice = self.flux_ratios[:, mask]
                e_slice = self.flux_ratios_err[:, mask]

                # calculate weighted mean for error handling
                weights = 1.0 / (e_slice ** 2)
                weighted_flux = np.sum(f_slice * weights, axis=1) / np.sum(weights, axis=1)
                weighted_err = np.sqrt(1.0 / np.sum(weights, axis=1))

                new_flux.append(weighted_flux)
                new_err.append(weighted_err)

        # overwrite the old class attributes
        self.time = np.array(new_time)
        self.flux_ratios = np.column_stack(new_flux)
        self.flux_ratios_err = np.column_stack(new_err)

        # recalculate phases using saved parameters
        self.set_phase(self.period['P'], self.period['P_err'], self.transit['t0'], self.transit['t0_err'])
        print(f'Completed Binning. New phase curve length: {len(self.time)}')

    def scale_magnitudes_to_BD(self, mag, err):
        R_p = self.planet_radius
        R_p_err = self.planet_radius_err
        R_BD = 0.9

        mag_offset = 5 * np.log10(R_p/R_BD)
        scaled_mag = mag + mag_offset
        
        radius_err = (5 / np.log(10)) * (R_p_err / R_p)
        scaled_mag_err = np.sqrt(err**2 + radius_err**2)
        
        return scaled_mag, scaled_mag_err

    def load_secondary_eclipse(self, path, planet):
        df = pd.read_csv(path)
        df = df[df['Planet'] == planet].copy()
        self.SecEclipse = df
        return True

    def get_sec_mag(self, band, scaled=True):
        if self.SecEclipse is None:
            print('run load_secondary_eclipse()!!')
            return None
            
        df = self.SecEclipse
        try:
            mag = df[band].iloc[0]
        except:
            print(f'ERROR: No band named {band}')
            return None
            
        if scaled == True:
            scaled_mag, _ = self.scale_magnitudes_to_BD(mag, 0.0)
            return scaled_mag
            
        return mag

    def get_sec_CMD(self, b1, b2, b3, scaled=True):
        m1 = self.get_sec_mag(b1, scaled)
        m2 = self.get_sec_mag(b2, scaled)
        m3 = self.get_sec_mag(b3, scaled)

        col = m1 - m2
        return col, m3

    def get_sec_eclipse_bands(self):
        df = self.SecEclipse
        cols = df.iloc[0].dropna().index.tolist()
        cols = [c for c in cols if c != "Planet"]
        return cols

In [ ]:
# 1. initialize object and load the data array
wasp121b_phasecurves_path = "./data/phase_curves/WASP-121b_Fp_exoTEDRF_5pixelbins.npz"
WASP121b = Process_Phase_Curves(filePath=wasp121b_phasecurves_path)
WASP121b.get_data()

# 2. set the precise parameters for WASP-121b
WASP121b.set_distance(plx_mas=3.7996, plx_mas_err=0.0104)
WASP121b.set_phase(P=1.274925, P_err=1.5e-7, t0=60244.520345, t0_err=0.000032)
WASP121b.set_radius(radius=1.7420, radius_err=0.0060)

# 3. bin the raw phase curve
WASP121b.bin_phase_curve(nbins=30)

# 4. load external datasets
photometry_path = "./data/phase_curves/Photmetry_WASP-121b_Updated_30_Binned_2_SNR.csv"
abbies_secondary_eclipses = "./data/phase_curves/Abbie_Exoplanets.csv"
WASP121b.load_photometry(filepath=Photometry_path)
WASP121b.load_secondary_eclipse(path=abbies_secondary_eclipses, planet='WASP-121b')

# 5. extract specific magnitudes and colours for plotting
wasp121b_cmd = WASP121b.get_CMD('F140M', 'F187N', 'F140M')

# (example plotting code)
# import matplotlib.pyplot as plt
# plt.scatter(wasp121b_cmd['col'], wasp121b_cmd['mag'], c=wasp121b_cmd['phase'])
# plt.gca().invert_yaxis()
# plt.show()

In [ ]:
# run synthetic photometry on phase curves
my_filters = Filter_lib.get_filters_covering_range(0.6012608647346497, 2.830579733848572)

#properties of WASP-121 from literature
master_df = WASP121.get_phoenix_spectroscopy(my_filters, logg=4.251, feh=0.17, alpha=0.0)

filename = './WASP-121b Synthetic_Photometry.csv'
master_df.to_csv(filename, index=False)

print(f'Data Saved to {filename}')
print(master_df.head())

In [ ]:
def process_single_sonora_file(filename, target_dir, m, filters_to_run, filter_lib, spec_lib):
    # extract teff and logg from the filename
    match = re.search(r't(\d+)g(\d+)', filename)
    if not match:
        return None

    Teff = int(match.group(1))
    g_raw = int(match.group(2))
    
    # convert surface gravity
    if g_raw > 0:
        logg = np.log10(g_raw) + 2 
    else:
        logg = 0.0

    full_path = os.path.join(target_dir, filename)

    try:
        # load spectrum
        data = pd.read_csv(full_path, sep=r'\s+', comment='#', names=['wave', 'flux'], usecols=[0, 1], compression='infer', low_memory=False, skiprows=1)
        
        data['wave'] = pd.to_numeric(data['wave'], errors='coerce')
        data['flux'] = pd.to_numeric(data['flux'], errors='coerce')
        
        # convert units to cgs
        c_angstrom = 2.99792e18 
        wave_angstrom = data['wave'] * 10000 
        data['flux'] = data['flux'] * (c_angstrom / wave_angstrom**2)
        data = data.dropna()

        # fetch radius from evolutionary tracks
        radius = spec_lib.get_radius(Teff=Teff, logg=logg, m=m)

        row_data = {'filename': filename, 'Teff': Teff, 'logg': round(logg, 2), 'M_H': m, 'Radius': radius}
        
        # integrate across all selected filters
        for fName, sysName in filters_to_run:
            result = filter_lib.integrate_filter(data['wave'], data['flux'], fName, sysName)

            if result is None:
                row_data[f'{fName}'] = np.nan
                continue
            
            flux_int, flux_err = result
            
            if flux_int is not None and flux_int > 0:
                mag = filter_lib.get_magnitude(fName, flux_int, radius, sysName)
                row_data[f'{fName}'] = mag
            else:
                row_data[f'{fName}'] = np.nan

        return row_data

    except Exception as e:
        print(f"Error processing {filename}: {e}")
        return None

def generate_sonora_photometry(m, filter_lib, spec_lib, filters='all'):
    filters_to_run = []
    
    # scan filter directory for requested systems
    if filters.lower() == 'all': 
        for file in os.listdir(filter_lib.FilterPath):
            match_nircam = re.search(r'JWST_NIRCam\.(F\w+)\.dat', file)
            if match_nircam: filters_to_run.append((match_nircam.group(1), 'NIRCam'))

            match_miri = re.search(r'JWST_MIRI\.(F\w+)\.dat', file)
            if match_miri: filters_to_run.append((match_miri.group(1), 'MIRI'))
                
            match_2mass = re.search(r'2MASS_2MASS\.([A-Za-z]+)\.dat', file)
            if match_2mass: filters_to_run.append((match_2mass.group(1), '2MASS'))
                
            match_spitzer = re.search(r'Spitzer_IRAC\.(I\d)\.dat', file)
            if match_spitzer: filters_to_run.append((match_spitzer.group(1), 'Spitzer'))
    else:
        for file in os.listdir(filter_lib.FilterPath):
            match_miri = re.search(r'JWST_MIRI\.(F\w+)\.dat', file)
            if match_miri: filters_to_run.append((match_miri.group(1), 'MIRI'))

    # remove duplicate filters
    filters_to_run = sorted(list(set(filters_to_run)))
    print(f"Found {len(filters_to_run)} filters.")

    # set the directory for the chosen metallicity
    if m == '0.0':
        folder_name = "spectra_m0.0"
    else:
        folder_name = f"spectra_m{m}"
        
    target_dir = os.path.join(spec_lib.sonora_path, folder_name)

    if not os.path.exists(target_dir):
        print(f"Error: Metallicity folder not found: {target_dir}")
        return None
    
    sonora_files = [f for f in os.listdir(target_dir) if f.startswith('sp_t')]
    print(f"Processing {len(sonora_files)} Sonora models in parallel...")
    
    # run the massive grid calculation across all cpu cores
    results = Parallel(n_jobs=-1, verbose=5)(
        delayed(process_single_sonora_file)(f, target_dir, m, filters_to_run, filter_lib, spec_lib) 
        for f in sonora_files
    )

    # remove failures and sort the final dataframe
    clean_results = [r for r in results if r is not None] 
    df = pd.DataFrame(clean_results)
    
    if not df.empty:
        df = df.sort_values(by=['Teff', 'logg'])
        
    return df

In [ ]:
# creates synthetic photometry for the Sonora models of a given metalicity
metalicity = '0.0'
df_new_0 = generate_sonora_photometry(metalicity, filters='all')

In [ ]:
# plots Figure 2
# shows differnet Teff sonora models next to filter profiles
# wrap the plot in a clean function and pass the library objects in
def plot_sonora_with_filters(spec_lib, filter_lib):
    plt.style.use('default')
    plt.rcParams.update({
        'font.size': 18, 'axes.labelsize': 16, 
        'xtick.labelsize': 16, 'ytick.labelsize': 16, 
        'legend.fontsize': 16, 'axes.linewidth': 2
    })

    temps_to_plot = [1500, 1000]
    colours = ['navy', 'darkorange']

    fig, ax1 = plt.subplots(figsize=(12, 6))

    filters = {'F150W': 1.50, 'F200W': 2.00, 'F115W': 1.15, 'F277W': 2.77, 'F356W': 3.56, 'F444W': 4.44}

    for i, t in enumerate(temps_to_plot):
        # fetch the spectrum (the class already handles the unit conversion to cgs)
        df = spec_lib.get_spectrum(teff=t, logg=3.0)

        if df is not None:
            # mask the wavelengths to our region of interest
            mask = (df['wave'] > 0.8) & (df['wave'] < 5.5)
            flux_masked = df['flux'][mask]

            # smooth the spectrum so it displays cleanly
            window_size = 200
            flux_smoothed = flux_masked.rolling(window=window_size, center=True).mean()

            # plot the smooth line
            ax1.plot(df['wave'][mask], flux_smoothed, label=f'{t} K',
                     linewidth=2.5, color=colours[i], alpha=0.9, zorder=2)

    ax2 = ax1.twinx()
    
    # overlay the jwst filter transmission profiles
    for name, centre in filters.items():
        band = filter_lib.get_filter(name)
        if band is not None:
            ax2.fill_between(band['wave'], band['trans'], alpha=0.2, zorder=1)

            max_trans = band['trans'].max()
            ax2.text(centre, max_trans + 0.03, name, ha='center', va='bottom', fontsize=13, fontweight='bold', color='dimgrey')

    ax1.set_xlabel("Wavelength ($\mathbf{\mu}$m)", fontweight='bold')
    ax1.set_ylabel(r"Flux Density ($\mathbf{F_\lambda}$) [erg s$^{\mathbf{-1}}$ cm$^{\mathbf{-2}}$ $\mathbf{\AA^{-1}}$]", fontweight='bold')

    ax1.margins(x=0)
    ax1.set_xlim(0.8, 5.5)

    ax1.set_yscale('log')
    ax1.autoscale(axis='y')
    ax1.set_ylim(1e10, 1e16)
    ax1.minorticks_on() 

    ax2.set_ylim(0, 4.0) 
    ax2.set_yticks([])

    ax1.legend(loc='upper right', frameon=True, edgecolor='black', framealpha=1.0)

    # save the figure to the repository plots folder
    plt.savefig('../plots/Figure_2_Sonora_Flux_Filters.pdf', format='pdf', bbox_inches='tight', dpi=300)
    plt.show()

# execute the plotting function
plot_sonora_with_filters(spec_lib, Filter_lib)

In [ ]:
# plots Figure 3
# pass spec_lib and poly_model into the function
def compare_poly_to_sonora(spec_lib, poly_model, b1, b2, b3):

    plt.style.use('default')
    plt.rcParams.update({
        'font.size': 18, 'axes.labelsize': 20, 
        'xtick.labelsize': 16, 'ytick.labelsize': 16, 
        'legend.fontsize': 16, 'axes.linewidth': 2,
        'mathtext.fontset': 'dejavusans'
    })
    
    # map the labels to the class attributes
    df_sonora = {
        '+0.5': spec_lib.photometry_pos,
        '0.0': spec_lib.photometry,
        '-0.5': spec_lib.photometry_neg
    }
    
    labels = ['+0.5', '0.0', '-0.5']
    logs_to_plot = [3, 4, 5, 5.5]
    colours = plt.cm.copper(np.linspace(0, 0.9, len(logs_to_plot)))
    
    fig, axes = plt.subplots(1, 3, figsize=(22, 7), sharey=True)
    
    # restrict the empirical fit to the model boundaries
    all_son_mag = []
    for label in labels:
        df_sub = df_sonora[label]
        all_son_mag.append(df_sub[b3].min())
    
    son_mag_min_global = min(all_son_mag)
    
    # extract the polynomial track arrays
    bd_col, bd_mag, spt = poly_model.for_CMD(b1, b2, b3)

    bd_mask = bd_mag >= son_mag_min_global
    bd_col = bd_col[bd_mask]
    bd_mag = bd_mag[bd_mask]
    spt = spt[bd_mask]

    # set plot boundaries with a small padding
    y_max = max(bd_mag) + 0.5
    y_min = min(bd_mag) - 0.5
    x_max = max(bd_col) + 0.2
    x_min = min(bd_col) - 0.2

    for i, (ax, label) in enumerate(zip(axes, labels)):
        df_sub = df_sonora[label]
                
        for log, col in zip(logs_to_plot, colours):
            mask = df_sub['logg'] == log

            # calculate colour and magnitude for this gravity
            son_col = df_sub[mask][b1] - df_sub[mask][b2]
            son_mag = df_sub[mask][b3]
            
            # clip the model tracks to the plot boundaries
            mask_clip = (
                (son_mag >= y_min) & (son_mag <= y_max) &
                (son_col >= x_min) & (son_col <= x_max)
            )
        
            son_col = son_col[mask_clip]
            son_mag = son_mag[mask_clip]
            
            ax.plot(son_col, son_mag, color=col, linewidth=2, label=f'$\log(g)$ = {log}')
            
        # format the multi-coloured empirical track
        points = np.array([bd_col, bd_mag]).T.reshape(-1, 1, 2)
        segments = np.concatenate([points[:-1], points[1:]], axis=1)

        norm = plt.Normalize(vmin=0, vmax=35) 
        lc = LineCollection(segments, cmap='plasma', norm=norm, zorder=3)
        
        lc.set_array(spt)
        lc.set_linewidth(4) 
        
        sc = ax.add_collection(lc) 
        ax.plot([], [], color='purple', linewidth=4, label='Observed BD Fit')
        
        ax.set_title(f"Metallicity [M/H] = {label}", fontweight='bold', pad=15)
        ax.set_xlabel(f'{b1}-{b2} (mag)', fontweight='bold')
        ax.set_ylabel(rf'Absolute Magnitude($\mathbf{{M_{{{b3}}}}}$)', fontweight='bold')
        ax.grid(True, linestyle=':', alpha=0.5)

        if i == 0:
            ax.set_ylabel(rf'Absolute Magnitude ($\mathbf{{M_{{{b3}}}}}$)', fontweight='bold')
            ax.invert_yaxis()

    handles, legend_labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, legend_labels, 
               loc='upper center',          
               bbox_to_anchor=(0.5, -0.02), 
               ncol=5,                      
               frameon=False,               
               prop={'weight':'bold'})

    cbar = fig.colorbar(sc, ax=axes.ravel().tolist(), pad=0.02, aspect=30)
    cbar.set_label('Spectral Type', fontweight='bold', labelpad=15)
    
    tick_positions = [0, 5, 10, 15, 20, 25, 30, 35]
    tick_labels = ["M0", "M5", "L0", "L5", "T0", "T5", "Y0", "Y5"]
    
    forced_vmin = min(tick_positions)
    forced_vmax = max(tick_positions)
    sc.set_clim(forced_vmin, forced_vmax)
    
    valid_ticks = [t for t in tick_positions if forced_vmin <= t <= forced_vmax]
    valid_labels = [tick_labels[tick_positions.index(t)] for t in valid_ticks]
    
    cbar.set_ticks(valid_ticks)
    cbar.set_ticklabels(valid_labels)
    
    # save the figure to the repository plots folder
    plt.savefig('../plots/Sonora_vs_Poly_Metallicity.pdf', format='pdf', bbox_inches='tight', dpi=300)
    plt.show()

# execute the plotting function
compare_poly_to_sonora(spec_lib, BD_Poly, 'J', 'Ks', 'J')

In [ ]:
# plots Figure 4
# shows the shift in magnitude of the phase curve after scaling the radius
plt.style.use('default')
plt.rcParams.update({
    'font.size': 14, 
    'axes.labelsize': 16, 
    'xtick.labelsize': 14, 
    'ytick.labelsize': 14, 
    'axes.linewidth': 1.5,
    'legend.fontsize': 12
})

b1 = 'F150W2'

fig, ax = plt.subplots(figsize=(8, 5))

scaled = WASP121.get_magnitudes(b1, scaled=True)
normal = WASP121.get_magnitudes(b1, scaled=False)
ax.plot(scaled['phase'], scaled['mag'], 
        color='#d62728', linestyle='-', marker='o', markersize=6, 
        label='Scaled ($0.9\,R_J$ Baseline)')

ax.plot(normal['phase'], normal['mag'], 
        color='grey', linestyle='--', marker='o', markersize=5, alpha=0.6, 
        label='Unscaled (Inflated Radius)')

target_phase = 0.15
idx = (np.abs(scaled['phase'] - target_phase)).argmin()

x_pos = scaled['phase'].iloc[idx] if hasattr(scaled['phase'], 'iloc') else scaled['phase'][idx]
y_top = normal['mag'].iloc[idx] if hasattr(normal['mag'], 'iloc') else normal['mag'][idx]
y_bottom = scaled['mag'].iloc[idx] if hasattr(scaled['mag'], 'iloc') else scaled['mag'][idx]

delta_m = y_bottom - y_top

ax.annotate('', 
            xy=(x_pos, y_bottom), 
            xytext=(x_pos, y_top), 
            arrowprops=dict(arrowstyle="<->", color='black', lw=1.5))

ax.text(x_pos + 0.05, (y_top + y_bottom) / 2, 
        f'$\\Delta M \\approx {delta_m:.2f}$', 
        verticalalignment='center', 
        bbox=dict(boxstyle='round,pad=0.5', fc='white', ec='black', alpha=0.9),
        fontsize=14)

ax.set_xlabel('Orbital Phase')
ax.set_ylabel(f'{b1} Absolute Magnitude')
ax.legend(loc='upper right')
ax.grid(True, linestyle=':', alpha=0.6)
ax.invert_yaxis() # Remember to invert!

plt.tight_layout()
file_path = "/plots/Phase curve scaled to BD comparison.pdf"
plt.savefig(file_path, format='pdf', bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
# function to create CMD
# used to plot figure 5


plt.style.use('default')
plt.rcParams.update({
    'font.size': 18, 'axes.labelsize': 16, 
    'xtick.labelsize': 16, 'ytick.labelsize': 16, 'axes.linewidth': 1.5
})

# pass the library objects in as arguments
def plot_any_cmd(b1, b2, b3, spec_lib, poly_model, wasp_model, psr_model, Sonora=True, PSR=True, Poly=True, WASP=True):

    plt.figure(figsize=(10,8))
    col_label = f'{b1}-{b2}'
    mid_color = plt.cm.inferno(0.5)
    
    # get the current axes for linecollections and colorbars
    ax = plt.gca()
    
    if Poly:
        x_bd, y_bd, x_common = poly_model.for_CMD(b1, b2, b3)
        ax.plot(x_bd, y_bd, zorder=2, color='orange', linewidth=3, label='Empirical BDs')

    if Sonora:
        df_sonora = {
            '+0.5': spec_lib.photometry_pos,
            '0.0': spec_lib.photometry,
            '-0.5': spec_lib.photometry_neg
        }
    
        labels = ['+0.5'] 
        logs_to_plot = [3, 4, 5, 5.5]
        
        # solid distinct colours for the gravities
        colours = ['#a6cee3', '#1f78b4', '#b2df8a', '#33a02c'] 
        
        # temperatures to highlight
        target_temps = [1200, 1600, 2000] 
    
        for label in labels:
            df_sub = df_sonora[label]

            isotherm_coords = {temp: {'x': [], 'y': []} for temp in target_temps}
            
            for log, col in zip(logs_to_plot, colours):
                mask = df_sub['logg'] == log

                # extract data arrays safely
                son_col_vals = (df_sub[mask][b1] - df_sub[mask][b2]).values
                son_mag_vals = df_sub[mask][b3].values
                son_teff_vals = df_sub[mask]['Teff'].values 
                
                # plot the solid background track
                my_label = f'$\log(g)$ = {log}' if log in [3, 5.5] else None
                ax.plot(son_col_vals, son_mag_vals, color=col, linewidth=2, 
                        alpha=0.7, zorder=1, label=my_label)
                
                # add the temperature markers
                for temp in target_temps:
                    # find closest temperature index
                    idx = (np.abs(son_teff_vals - temp)).argmin()
                    x_mark = son_col_vals[idx]
                    y_mark = son_mag_vals[idx]
                    
                    # plot a square marker
                    ax.plot(x_mark, y_mark, marker='s', color=col, markersize=5, zorder=2)

                    isotherm_coords[temp]['x'].append(x_mark)
                    isotherm_coords[temp]['y'].append(y_mark)
                    
                    # add text label only to highest gravity track
                    if log == 5.5: 
                        if temp == 2000:
                            ax.text(x_mark + 0.047, y_mark + 0.2, f'{temp} K', fontsize=12, 
                                fontweight='bold', color='grey', ha='right', va='center')
                        else:
                            ax.text(x_mark - 0.025, y_mark, f'{temp} K', fontsize=12, 
                                    fontweight='bold', color='grey', ha='right', va='center')
                                    
            # plot the dashed isotherms linking the markers
            for temp in target_temps:
                ax.plot(isotherm_coords[temp]['x'], isotherm_coords[temp]['y'], 
                        color='dimgrey', linestyle=':', linewidth=1.5, alpha=0.5, zorder=1)

    # shared phase parameters for exoplanet phase curves
    phase_cmap = 'inferno'
    phase_vmin = -0.5 
    phase_vmax = 0.5

    if WASP:
        wasp_data = wasp_model.get_CMD(b1, b2, b3)
        sc_wasp = ax.scatter(wasp_data['col'], wasp_data['mag'], c=wasp_data['phase'], 
                             cmap=phase_cmap, vmin=phase_vmin, vmax=phase_vmax, 
                             marker='o', s=80, edgecolor='black', linewidth=0.5, 
                             zorder=4)
        ax.scatter([], [], color=mid_color, marker='o', s=80, edgecolor='black', label='WASP-121b')
    
    if PSR:
        psr_cmd = psr_model.get_CMD(b1, b2, b3)
        sc_psr = ax.scatter(psr_cmd['col'], psr_cmd['mag'], c=psr_cmd['phase'], 
                            cmap=phase_cmap, vmin=phase_vmin, vmax=phase_vmax, 
                            marker='^', s=120, edgecolor='black', linewidth=0.5, 
                            zorder=5)
        ax.scatter([], [], color=mid_color, marker='^', s=80, edgecolor='black', label='PSR J2322-2650b')
        
    # add a shared colour bar
    if WASP or PSR:
        cbar = plt.colorbar(sc_wasp if WASP else sc_psr, ax=ax, pad=0.02)
        cbar.set_label('Orbital Phase', rotation=270, labelpad=20)

    ax.invert_yaxis()
    
    # format legend
    ax.legend(loc='lower center', bbox_to_anchor=(0.5, 1.02), ncol=4, 
          frameon=False, fontsize=12, handletextpad=0.5, columnspacing=1.5)
    
    plt.xlim(0.5, 2.5)
    plt.ylim(16.5, 10.8)
    plt.ylabel(rf'Absolute Magnitude ($M_{{{b3}}}$)')
    plt.xlabel(f'{col_label} (mag)')
    
    # dynamically build the filename and save to the plots folder
    met_str = labels[0] if Sonora else "NoSonora"
    filename = f'../plots/CMD_{b3}_vs_{col_label}_met_{met_str}.pdf'
    
    plt.savefig(filename, format='pdf', bbox_inches='tight', dpi=300)
    plt.show()

# run the plotting function
b1 = 'F140M'
b2 = 'Ks'
b3 = b1
plot_any_cmd(b1, b2, b3, spec_lib, BD_Poly, WASP121, psr, Poly=False)

In [ ]:
# plots Figure 7


plt.style.use('default')
plt.rcParams.update({
    'font.size': 18, 'axes.labelsize': 16, 
    'xtick.labelsize': 16, 'ytick.labelsize': 16, 'axes.linewidth': 1.5
})

# pass the library objects in as arguments
def plot_any_cmd_with_cbar(b1, b2, b3, spec_lib, poly_model, wasp_model, psr_model, met='0.0', Sonora=True, PSR=True, Poly=True, WASP=True):

    plt.figure(figsize=(10,7))
    col_label = f'{b1}-{b2}'
    mid_color = plt.cm.inferno(0.5)
    
    ax = plt.gca()
    
    # create variables to store the mapped colours for later
    poly_mappable = None
    phase_mappable = None
    
    if Poly:
        x_bd, y_bd, spt = poly_model.for_CMD(b1, b2, b3)

        points = np.array([x_bd, y_bd]).T.reshape(-1, 1, 2)
        segments = np.concatenate([points[:-1], points[1:]], axis=1)

        # changed cmap to 'viridis' so it contrasts with the phase curves
        norm = plt.Normalize(vmin=0, vmax=35) 
        lc = LineCollection(segments, cmap='viridis', norm=norm, zorder=2)
        
        lc.set_array(spt)
        lc.set_linewidth(3.5) 
        
        # save the line to our variable instead of drawing the colour bar right now
        poly_mappable = ax.add_collection(lc) 
        
        mid_poly_colour = plt.cm.viridis(0.5)
        ax.plot([], [], color=mid_poly_colour, linewidth=3.5, label='Empirical Sequence')

    if Sonora:
        df_sonora = {
            '+0.5': spec_lib.photometry_pos,
            '0.0': spec_lib.photometry,
            '-0.5': spec_lib.photometry_neg
        }
    
        labels = [met] 
        logs_to_plot = [3, 4, 5, 5.5]
        colours = ['#a6cee3', '#1f78b4', '#b2df8a', '#33a02c'] 
        target_temps = [500, 600, 700, 800, 1000, 1200, 1400]
    
        for label in labels:
            df_sub = df_sonora[label]
            isotherm_coords = {temp: {'x': [], 'y': []} for temp in target_temps}
            
            for log, col in zip(logs_to_plot, colours):
                mask = df_sub['logg'] == log

                son_col_vals = (df_sub[mask][b1] - df_sub[mask][b2]).values
                son_mag_vals = df_sub[mask][b3].values
                son_teff_vals = df_sub[mask]['Teff'].values 
                
                my_label = f'$\log(g)$ = {log}' if log in [3, 5.5] else None
                ax.plot(son_col_vals, son_mag_vals, color=col, linewidth=2, 
                        alpha=0.7, zorder=1, label=my_label)
                
                for temp in target_temps:
                    idx = (np.abs(son_teff_vals - temp)).argmin()
                    x_mark = son_col_vals[idx]
                    y_mark = son_mag_vals[idx]
                    
                    ax.plot(x_mark, y_mark, marker='s', color=col, markersize=5, zorder=2)

                    isotherm_coords[temp]['x'].append(x_mark)
                    isotherm_coords[temp]['y'].append(y_mark)
                    
                    if log == 5.5: 
                        if temp == 2000:
                            ax.text(x_mark + 0.047, y_mark + 0.2, f'{temp} K', fontsize=12, 
                                fontweight='bold', color='grey', ha='right', va='center')
                        else:
                            ax.text(x_mark + 0.08, y_mark + 0.2, f'{temp} K', fontsize=12, 
                                    fontweight='bold', color='grey', ha='right', va='center')
            for temp in target_temps:
                ax.plot(isotherm_coords[temp]['x'], isotherm_coords[temp]['y'], 
                        color='dimgrey', linestyle=':', linewidth=1.5, alpha=0.5, zorder=1)

    phase_cmap = 'inferno'
    phase_vmin = -0.5 
    phase_vmax = 0.5

    if WASP:
        wasp_data = wasp_model.get_CMD(b1, b2, b3)
        # save to phase_mappable
        phase_mappable = ax.scatter(wasp_data['col'], wasp_data['mag'], c=wasp_data['phase'], 
                             cmap=phase_cmap, vmin=phase_vmin, vmax=phase_vmax, 
                             marker='o', s=80, edgecolor='black', linewidth=0.5, 
                             zorder=4)
        ax.scatter([], [], color=mid_color, marker='o', s=80, edgecolor='black', label='WASP-121b')
    
    if PSR:
        psr_cmd = psr_model.get_CMD(b1, b2, b3)
        # save to phase_mappable
        phase_mappable = ax.scatter(psr_cmd['col'], psr_cmd['mag'], c=psr_cmd['phase'], 
                            cmap=phase_cmap, vmin=phase_vmin, vmax=phase_vmax, 
                            marker='^', s=120, edgecolor='black', linewidth=0.5, 
                            zorder=5)
        ax.scatter([], [], color=mid_color, marker='^', s=80, edgecolor='black', label='PSR J2322-2650b')
        
    # build the colour bars here
    
    # draw phase first
    if WASP or PSR:
        cbar_phase = plt.colorbar(phase_mappable, ax=ax, pad=0.02)
        cbar_phase.set_label('Orbital Phase', rotation=270, labelpad=20)

    # draw spectral type second
    if Poly and poly_mappable is not None:
        cbar_spt = plt.colorbar(poly_mappable, ax=ax, pad=0.1) 
        cbar_spt.set_label('Spectral Type', rotation=270, labelpad=20)
        
        tick_positions = [0, 5, 10, 15, 20, 25, 30, 35]
        tick_labels = ["M0", "M5", "L0", "L5", "T0", "T5", "Y0", "Y5"]
        cbar_spt.set_ticks(tick_positions)
        cbar_spt.set_ticklabels(tick_labels)

    ax.invert_yaxis()
    
    ax.legend(loc='upper right', frameon=True, framealpha=0.9, edgecolor='black', fontsize=12)
    
    plt.ylabel(rf'Absolute Magnitude ($M_{{{b3}}}$)')
    plt.xlabel(f'{col_label} (mag)')

    plt.xlim(2, 6)
    plt.ylim(15, 9)
    plt.minorticks_on()

    plt.grid(which='minor', linestyle=':', linewidth=0.5, alpha=0.4)
    
    # dynamically build the filename and save to the plots folder
    met_str = met if Sonora else "NoSonora"
    filename = f'../plots/CMD_DoubleCBar_{b3}_vs_{col_label}_met_{met_str}.pdf'
    plt.savefig(filename, format='pdf', bbox_inches='tight', dpi=300)
    
    plt.show()

# run the plotting function
b1 = 'J'
b2 = 'Ks'
b3 = b1
# plot_any_cmd_with_cbar(b1, b2, b3, spec_lib, BD_Poly, WASP121, psr, met='+0.5', Poly=True, Sonora=False)

In [ ]:
b1 = 'J'
b2 = '[4.5]'
b3 = b2
plot_any_cmd_with_cbar(b1, b2, b3, met='-0.5', Poly=False, WASP=False)

In [ ]:
b1 = 'F115W'
b2 = '[4.5]'
b3 = '[4.5]'
plot_any_cmd_with_cbar(b1, b2, b3, met='-0.5', Poly=False, WASP=False)

In [ ]:
b1 = 'F115W'
b2 = 'F444W'
b3 = b2
Plot_any_CMD1(b1, b2, b3, met='-0.5', Poly=False, WASP=False)

In [ ]:
# plots Figure 8
# compares all filter bandpasses used in project

# wrap the plot in a function and pass the filter library in
def plot_filter_bandpasses(filter_lib):
    fig, axes = plt.subplots(nrows=4, ncols=1, figsize=(12, 10), sharex=True)
    graphs = ['W2', 'W', 'M', 'Reference']
    system = 'JWST'
    row_titles = {
        'W2': 'Wide²',
        'W': 'Wide Bands',
        'M': 'Medium Bands',
        'Reference': '2MASS + Spitzer'
    }
    
    for i, ax in enumerate(axes):
        category = graphs[i]
        ax.text(0.02, 0.85, row_titles.get(category, category), transform=ax.transAxes, 
                fontsize=12, fontweight='bold', bbox=dict(facecolor='white', alpha=0.8, edgecolor='none'))
        
        if category == 'Reference':
            mass_files = filter_lib.get_width_files('2MASS', system='2MASS')
            if mass_files:
                for fName in mass_files:
                    band = filter_lib.get_filter(fName, system='2MASS')
                    if band is not None:
                        ax.fill_between(band['wave'], band['trans'], color='green', alpha=0.3)
                        midpoint = filter_lib.get_midpoint(fName, '2MASS')
                        ax.text(midpoint, 0.3, fName, ha='center', va='bottom', 
                                transform=ax.get_xaxis_transform(), fontsize=10, color='black')
                        ax.minorticks_on()
                        ax.xaxis.set_major_locator(ticker.MultipleLocator(0.5))
                        
            spitzer_files = filter_lib.get_width_files('Spitzer', system='Spitzer')
            if spitzer_files:
                for fName in spitzer_files:
                    band = filter_lib.get_filter(fName, system='Spitzer')
                    if band is not None:
                        ax.fill_between(band['wave'], band['trans'], color='orange', alpha=0.3)
                        midpoint = filter_lib.get_midpoint(fName, 'Spitzer')
                        
                        # relabel the specific spitzer bands for clarity
                        if fName == 'I1': 
                            fName = '[3.6]'
                            midpoint = 3.55
                        elif fName == 'I2': 
                            fName = '[4.5]'
                            
                        ax.text(midpoint, 0.2, fName, ha='center', va='bottom', 
                                transform=ax.get_xaxis_transform(), fontsize=10, color='black')
                        ax.minorticks_on()
                        ax.xaxis.set_major_locator(ticker.MultipleLocator(0.5))
        else:
            filterNames = filter_lib.get_width_files(category, system)
            if filterNames is not None:
                for filterName in filterNames:
                    band = filter_lib.get_filter(filterName, system)
                    midpoint = filter_lib.get_midpoint(filterName, system)
                    ax.fill_between(band['wave'], band['trans'], alpha=0.3)
                    ax.text(midpoint, 0.3, filterName, ha='center', va='bottom', 
                            transform=ax.get_xaxis_transform(), fontsize=10, color='black', rotation=90)
                    ax.minorticks_on()
                    ax.xaxis.set_major_locator(ticker.MultipleLocator(0.5))
                    
    # add the vertical marker lines
    vert_lines = [3.13, 3.94, 5.05]
    for ax in axes:
        for x in vert_lines:
            ax.axvline(x, color='red', linestyle='--', alpha=0.7)

    fig.supxlabel(r'Wavelength ($\boldsymbol{\mu m}$)', fontsize=16, fontweight='bold', y=0.05)
    fig.supylabel('Transmission', fontsize=16, fontweight='bold', x=0.05)
    fig.suptitle('JWST NIRCam Filters + 2MASS + Spitzer', fontsize=18, fontweight='bold', y=0.91)
    
    # save the figure directly to the plots folder
    plt.savefig('../plots/Figure_8_Filter_Bandpasses.pdf', format='pdf', bbox_inches='tight', dpi=300)
    plt.show()

# execute the plotting function
plot_filter_bandpasses(Filter_lib)

In [ ]:
# plots Figure 9 in appendix B
#compares blackbody approx vs PHOENIX model for WASP-121b
# wrap the plot in a function and pass the wasp121 object in
def plot_stellar_flux_comparison(wasp121):
    # define the stellar parameters for wasp-121
    star_teff = 6460
    star_teff_err = 140
    star_radius = 1.458
    star_radius_err = 0.030
    star_logg = 4.24
    star_feh = 0.13
    star_alpha = 0.0

    # get the blackbody flux using the new function arguments
    bb_flux, bb_err = wasp121.get_stellar_flux(wasp121.wavelengths, star_teff, star_teff_err, star_radius, star_radius_err)

    # get the phoenix model flux
    phoenix_flux, _ = wasp121.phoenix_models(wasp121.wavelengths, star_logg, star_feh, star_alpha, star_teff, star_radius)

    plt.style.use('default')
    plt.rcParams.update({
        'font.size': 16, 'axes.labelsize': 16, 
        'xtick.labelsize': 14, 'ytick.labelsize': 14, 'axes.linewidth': 2
    })

    plt.figure(figsize=(12, 6))

    # plot the smooth blackbody
    plt.plot(wasp121.wavelengths, bb_flux, label=r"Blackbody Approximation ($T_{\rm eff} = 6460\,$K)", 
             color='orange', linestyle='--', linewidth=2.5, zorder=2)

    # plot the detailed phoenix model
    plt.plot(wasp121.wavelengths, phoenix_flux, label="PHOENIX Stellar Atmosphere", 
             color='navy', linewidth=1.5, alpha=0.9, zorder=1)

    # formatting
    plt.xlabel(r"Wavelength ($\mu$m)")
    plt.ylabel(r"Flux at Earth (erg s$^{-1}$ cm$^{-2}$ $\AA^{-1}$)")

    plt.legend(loc='upper right', framealpha=1.0, edgecolor='black')
    plt.grid(True, linestyle=':', alpha=0.5)
    plt.minorticks_on()

    # save the figure directly to the plots folder
    plt.savefig('../plots/Appendix_WASP121_Stellar_Flux_Comparison.pdf', format='pdf', bbox_inches='tight', dpi=300)
    plt.show()

# execute the plotting function
plot_stellar_flux_comparison(WASP121)